# Unitree G1 MuJoCo PPO Training - Clean V2

This notebook is the clean, current version of the G1 MuJoCo training flow.

Goal: train a prototype PPO policy for Unitree G1 lower-body locomotion in MuJoCo on Colab.

Important: this is a simulation learning prototype, not a deployable real-robot walking controller.

## 0. Runtime

Use `Runtime > Change runtime type > T4 GPU` if available. CPU also works for a smoke test, but training is slower.

## 1. Clone Fresh Helper Repo

In [ ]:
%cd /content
!rm -rf reacher-bc-robustness
!git clone https://github.com/vijayvanapalli96/reacher-bc-robustness.git
%cd reacher-bc-robustness
!git log -1 --oneline

## 2. Install Dependencies

In [ ]:
!pip -q install mujoco==3.2.3 stable-baselines3 torch pyyaml imageio tensorboard pandas matplotlib

## 3. Clone Unitree RL Gym

In [ ]:
!rm -rf unitree_rl_gym
!git clone --depth 1 https://github.com/unitreerobotics/unitree_rl_gym.git
!ls unitree_rl_gym/deploy/deploy_mujoco/configs
!ls unitree_rl_gym/deploy/pre_train/g1

## 4. MuJoCo G1 Smoke Test

This only loads the G1 XML and steps physics with a standing PD controller. If this fails, training will fail too.

In [ ]:
!python -u scripts/unitree_g1_mujoco_smoke_test.py --unitree-root unitree_rl_gym --steps 100

## 5. PPO Environment Check

You want to see `initial_upright_score` near `1.000` and `terminated: False`.

In [ ]:
!python -u scripts/unitree_g1_mujoco_ppo_train.py --unitree-root unitree_rl_gym --timesteps 1 --num-envs 1 --check-env

## 6. Train PPO Without Rendering

First run `100000` timesteps. Watch `ep_len_mean`. If it is much larger than `1`, the environment is no longer immediately terminating.

In [ ]:
!mkdir -p artifacts
!python -u scripts/unitree_g1_mujoco_ppo_train.py --unitree-root unitree_rl_gym --timesteps 100000 --num-envs 4 --episode-seconds 6 --no-render 2>&1 | tee artifacts/g1_mujoco_ppo_train.log

## 7. Render The Trained Policy

Rendering is separated from training so OpenGL issues do not kill the training run.

In [ ]:
!MUJOCO_GL=egl python -u scripts/unitree_g1_render_trained_ppo.py \
  --model artifacts/g1_mujoco_ppo/ppo_g1_mujoco_final.zip \
  --unitree-root unitree_rl_gym \
  --out artifacts/g1_mujoco_ppo/ppo_g1_mujoco_rollout.gif \
  --duration 6.0

## 8. Display The GIF

In [ ]:
from IPython.display import Image, display
display(Image("artifacts/g1_mujoco_ppo/ppo_g1_mujoco_rollout.gif"))

## 9. Show Final Training Lines

Paste this output back to Codex.

In [ ]:
!tail -n 80 artifacts/g1_mujoco_ppo_train.log

## What To Report Back

Tell Codex:

1. Did the smoke test pass?
2. In the PPO check, was `terminated` false?
3. What was the final `ep_len_mean`?
4. Did the GIF show falling, standing, shuffling, or walking?